# PINN4SOH 模块 4：三项 Loss 设计

### 数据流中的位置

```
[模块3: PINN.forward()]
    │
    ├── u₁, f₁ = forward(x₁)     ← F(·): Encoder→Predictor
    ├── u₂, f₂ = forward(x₂)     ← G(·): dynamical_F
    │
    ▼
[模块4: 本模块 — 三项 Loss]
    │
    ├── L_data  = ½MSE(u₁,y₁) + ½MSE(u₂,y₂)     ← 数据保真度
    ├── L_PDE   = ½MSE(f₁,0)  + ½MSE(f₂,0)      ← 物理一致性
    ├── L_mono  = Σ ReLU((u₂-u₁)×(y₁-y₂))        ← 单调性约束
    └── L_total = L_data + α·L_PDE + β·L_mono
    │
    ▼
[模块5: 训练循环] ← loss.backward() + optimizer.step()
```

---

## 复现运行与源码对齐说明

> 当前 clean 代码：`src/04_loss.py`  
> 原始代码：`Model/Model.py L237-L275`  
> 执行规范：paper-reproduction；原始仓库只读。  
> 本手册已完成 Notebook JSON 与代码单元静态检查。涉及数据的单元需按 `DATA.md` 准备数据后，从头运行以生成本机结果。

本手册保留六层学习结构：纯净代码、逐行详解、语法表、数据流角色、物理含义与真实数据图、踩坑记录。论文与源码不一致处以源码为复现基准并单独说明。

### 背景：为什么 PINN 需要三项 Loss？

**纯数据驱动的神经网络**只有一个目标：让预测值接近真实值 (MSE)。这在小数据集上容易过拟合，在电池退化预测上尤其危险——模型可能学出"SOH 先降后升"这种违反物理规律的曲线。

**PINN 的核心创新**是在 Loss 中注入物理知识：

| Loss | 注入什么物理知识 | 没有它会怎样 |
|------|-----------------|-------------|
| L_data | 预测要准 | 模型什么都不学 |
| L_PDE | SOH 退化遵循 du/dt = G(x,u) | 模型可能学出非物理解（如震荡、跳变） |
| L_mono | 电池 SOH 只减不增（容量回升除外） | 模型可能产生缺乏依据的 SOH 上升；真实数据仍可能因测量噪声、容量恢复等出现局部回升 |

**类比理解**：L_data 是"考试要对答案"，L_PDE 是"解题过程要符合公式"，L_mono 是"答案不能违反常识"。三道关卡一起把关，才能在小样本下学到可靠模型。

### 模块 4 核心概念概览

> 行号记录自手册编写版本，后续整理可能产生偏移；核对实现时请以函数名和当前 `src/` 文件为准。

| 子模块 | clean_code 行号 | 原始代码行号 | 核心变换 |
|--------|:-----------:|:----------:|----------|
| 辅助: AverageMeter | L13-31 | `utils/util.py` L23-39 | 累计每个 epoch 的三项平均 Loss |
| 4a: L_data | L34-36 | L247-248 | (u₁,u₂,y₁,y₂) → 两个 MSE 取平均 |
| 4b: L_PDE | L39-42 | L250-252 | (f₁,f₂) → 让 PDE 残差逼近 0 |
| 4c: L_mono ⭐ | L45-47 | L254-255 | (u₁,u₂,y₁,y₂) → ReLU 惩罚方向错误 |
| 4d: L_total + α,β | L50-57 | L257-258 | 三项加权求和 |

### 目录

| 章节 | clean_code 行号 | 原始代码行号 | 内容 |
|------|:----------:|:----------:|------|
| [0. AverageMeter — Loss 统计辅助工具](#sec0) | 13-31 | `utils/util.py` 23-39 | 记录当前值、累计和与 epoch 平均值 |
| [1. 4a: L_data — 数据保真度损失](#sec1) | 34-36 | 247-248 | 为什么用 0.5×两个MSE？ |
| [2. 4b: L_PDE — PDE 残差损失](#sec2) | 39-42 | 250-252 | f=0 的物理含义 |
| [3. 4c: L_mono — 单调性约束 ⭐](#sec3) | 45-47 | 254-255 | ReLU trick 的巧妙之处 |
| [4. 4d: α, β 参数分析](#sec4) | 50-57 | 257-258 | 权重如何影响训练行为 |
| [5. 完整训练一步](#sec5) | 60-81 | 237-275 | train_one_epoch 串联全部 Loss |

<a id="sec0"></a>
### 0. AverageMeter：Loss 统计辅助工具

> **对应 src/04_loss.py 第 13-31 行**  
> **对应原始代码 utils/util.py 第 23-39 行；在 Model/Model.py 第 239-241、266-275 行使用**

`AverageMeter` 来自原始训练流程，用于把一个 epoch 中多个 batch 的 Loss 累计成平均值。它是**训练监控和日志统计环节**，不参与 Loss 公式、自动微分、反向传播或模型参数更新。

```python
class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0.0
        self.avg = 0.0
        self.sum = 0.0
        self.count = 0

    def update(self, value, count=1):
        self.val = value
        self.sum += value * count
        self.count += count
        self.avg = self.sum / self.count
```

| 属性 | 含义 |
|------|------|
| `val` | 最近一个 batch 的 Loss |
| `sum` | 目前累计的加权 Loss 总和 |
| `count` | 累计权重；当前训练代码每个 batch 默认加 1 |
| `avg` | 当前累计平均值，即 `sum / count` |

在 `train_one_epoch()` 中，三个实例分别统计数据 Loss、PDE Loss 和单调性 Loss：

```python
meters = [AverageMeter(), AverageMeter(), AverageMeter()]
for meter, value in zip(meters, [data_loss, pde_loss, mono_loss]):
    meter.update(value.item())
return tuple(meter.avg for meter in meters)
```

例如三个 batch 的某项 Loss 依次为 `0.10、0.08、0.06`，更新后 `val=0.06`、`sum=0.24`、`count=3`、`avg=0.08`。删除 `AverageMeter` 不会改变模型训练结果，但会失去每个 epoch 的平均三项 Loss 记录。

> ⚠️ 当前源码调用 `update(value.item())`，所以得到的是**按 batch 等权平均**。如果要严格按样本数加权，应调用 `update(value.item(), x1.shape[0])`；本复现保留原始代码行为。

---

## 1. 4a: L_data — 数据保真度损失 <a id="sec1"></a>

> **对应 src/04_loss.py 第 34-36 行**
> **对应原始代码 Model/Model.py 第 247-248 行**

一句话：让模型在两个时刻的 SOH 预测都接近真实值。这是所有监督学习的共同基础。

### 1.1 代码块

In [ ]:
import torch
import torch.nn as nn
import numpy as np

def compute_data_loss(u1, u2, y1, y2, loss_func):
    """
    L_data: 数据保真度损失

    公式: L_data = 0.5 * MSE(u1, y1) + 0.5 * MSE(u2, y2)

    参数:
        u1, u2: 模型预测的 SOH 值 (batch_size, 1)
        y1, y2: 真实的 SOH 值 (batch_size, 1)
        loss_func: nn.MSELoss()

    物理含义:
        0.5 系数让两个时刻的 MSE 权重均等。
        若只用 u1 算 loss → 模型只关心"当前时刻 SOH"
        若只用 u2 算 loss → 模型只关心"下一时刻 SOH"
        两者各取一半 → 同时关心两个时刻的预测精度。
    """
    return 0.5 * loss_func(u1, y1) + 0.5 * loss_func(u2, y2)

### 1.1.1 逐行详解

```python
def compute_data_loss(u1, u2, y1, y2, loss_func):
#   └──────┬──────┘ └┘ └┘ └┘ └┘ └───┬───┘
#    [函数定义]   4个预测/真实张量  [MSE损失函数]
```

```python
    return 0.5 * loss_func(u1, y1) + 0.5 * loss_func(u2, y2)
    #      └┬┘   └────┬────┘         └────┬────┘
    #    [权重½] [MSE(u₁,真值₁)]    [MSE(u₂,真值₂)]
    #
    #  为什么是 0.5 而不是 1.0？
    #  MSE(u₁,y₁) 和 MSE(u₂,y₂) 各自已经是对 batch 的平均
    #  0.5×(两者之和) = 两者的算术平均 = 每个时刻权重相等
    #  如果不用 0.5 → L_data 大约是其他 Loss 的 2 倍
    #  → 需要调大 α 和 β 来平衡 → 增加了调参复杂度
```

**关键理解：数据流形状**

```
x1: (batch, 17) → PINN.forward(x1) → u1: (batch, 1)
x2: (batch, 17) → PINN.forward(x2) → u2: (batch, 1)
y1: (batch, 1)                      y2: (batch, 1)

L_data = 0.5 * mean((u1 - y1)²) + 0.5 * mean((u2 - y2)²)
```

两个时刻独立计算 MSE，谁也不依赖谁。但它们的梯度会通过反向传播汇合到同一个 Solution_u 网络中。

### 1.2 语法速查表

| 语法点 | 含义 | 这里为什么用 |
|--------|------|-------------|
| `nn.MSELoss()` | 均方误差: mean((pred-true)²) | 回归任务的标准损失，对离群值敏感（平方放大） |
| `0.5 * (A + B)` | A和B的算术平均 | 两个时刻权重均等，防止某时刻主导梯度 |
| 独立计算再相加 | 梯度累加 | 两个 loss 共享同一个计算图，backward 时梯度自动汇合 |

### 1.3 逻辑：这一步在整体流程中的角色

```
u₁,u₂ (预测) ──┐
               ├── L_data = ½MSE(u₁,y₁) + ½MSE(u₂,y₂)
y₁,y₂ (真实) ──┘
               │
               ▼
          L_data 是最"直接"的监督信号
          没有它 → 模型完全不学习 SOH 预测
```

**为什么 L_data 不能单独使用？** 单独 L_data = 普通 MLP。在小样本电池数据上，纯数据驱动容易过拟合——模型记住训练电池的退化轨迹，但在新电池上泛化很差。L_PDE 和 L_mono 就是为了解决这个问题而加的。

### 1.4 物理含义

L_data 的物理含义最直观：**"预测要准"**。

- L_data = 0.01 → 平均每个预测偏离真实值约 √0.01 = 0.1 SOH 单位 → 约 10% 误差（未训练的初始状态）
- L_data = 0.0001 → 平均偏离约 0.01 SOH → 约 1% 误差（良好训练后）
- L_data = 0.000001 → 平均偏离约 0.001 SOH → 约 0.1% 误差（可能过拟合）

### 1.5 踩坑记录

| 错误做法 | 现象 | 正确做法 |
|----------|------|----------|
| 用 L1Loss (MAE) 代替 MSE | 对大误差不敏感，训练慢 | 保持 MSE（平方对大误差梯度更大→学得更快） |
| 只用 u₁ 算 loss | 模型不关心下一时刻预测，L_PDE 无法约束退化方向 | 必须 u₁ 和 u₂ 各算一份 |

## 2. 4b: L_PDE — PDE 残差损失 <a id="sec2"></a>

> **对应 src/04_loss.py 第 39-42 行**
> **对应原始代码 Model/Model.py 第 250-252 行**

一句话：强制模型学到的 SOH 退化过程满足微分方程 du/dt = G(x, u, u_x, u_t)。这是 **PINN 区别于普通 MLP 的核心标志**。

### 2.1 代码块

In [ ]:
def compute_pde_loss(f1, f2, loss_func):
    """
    L_PDE: PDE 残差损失（物理约束）

    公式: L_PDE = 0.5 * MSE(f1, 0) + 0.5 * MSE(f2, 0)

    f = u_t - G(x, u, u_x, u_t) 是 PDE 残差。
    如果电池退化严格满足退化方程 du/dt = G(...)，则 f 应为 0。

    物理含义:
        f=0 意味着"预测的退化速率 = 学到的物理规律预测的退化速率"
        即 du/dt = G(...)，退化过程严格遵循学到的物理规律。
        L_PDE 越小 → 模型越"物理一致"。
    """
    f_target = torch.zeros_like(f1)
    return 0.5 * loss_func(f1, f_target) + 0.5 * loss_func(f2, f_target)

### 2.1.1 逐行详解

```python
def compute_pde_loss(f1, f2, loss_func):
    #  └──────┬──────┘ └┘ └┘ └───┬───┘
    #   [函数定义]  PDE残差    [MSE loss]
```

```python
    f_target = torch.zeros_like(f1)
    #          └──────┬──────┘
    #    [创建与 f1 同形状的全零张量]
    #    代表"理想 PDE 残差 = 0"
    #    即退化过程完全满足物理方程
```

```python
    return 0.5 * loss_func(f1, f_target) + 0.5 * loss_func(f2, f_target)
    #      └┬┘   └────────┬───────┘         └────────┬───────┘
    #    [权重½]  MSE(f₁, 0) → 让 f₁ 逼近 0    MSE(f₂, 0) → 让 f₂ 逼近 0
```

**f = u_t - F 是怎么来的？**

回顾模块 3 中 PINN.forward() 的关键操作：
```python
u_t = grad(u.sum(), t, create_graph=True)[0]   # autograd 自动求导
u_x = grad(u.sum(), x, create_graph=True)[0]   # 对 16 个特征求偏导
F = self.dynamical_F(torch.cat([xt, u, u_x, u_t], dim=1))
f = u_t - F
```

- u_t = ∂u/∂t: SOH 关于循环数 t 的变化率（退化速率），由 autograd 自动算出
- F = G(xt, u, u_x, u_t): dynamical_F 网络预测的"应有的退化速率"
- f = u_t - F: 如果两者相等 → f=0 → 退化过程满足物理规律

**核心洞见**：dynamical_F 在学习"电池退化应该有多快"。它学到的不是一个数，而是一个函数 G(x, u, u_x, u_t) → 退化速率。这个函数对所有电池、所有时刻都适用（如果学得好）。

### 2.2 语法速查表

| 语法点 | 含义 | 这里为什么用 |
|--------|------|-------------|
| `torch.zeros_like(f1)` | 创建与 f1 同形状的全零张量 | "理想状态"是 f=0（退化满足物理方程） |
| `MSE(f, 0)` | f 到 0 的均方距离 | 等价于 mean(f²)，让 f 在所有样本上都逼近 0 |
| `torch.cat([xt, u, u_x, u_t], dim=1)` | 沿特征维拼接 4 个张量 | dynamical_F 需要 17+1+16+1=35 维输入 |

**35 维输入的组成**：
| 来源 | 维度 | 含义 |
|------|:---:|------|
| xt (输入) | 17 | 16特征 + cycle_index |
| u (SOH预测) | 1 | 当前预测的健康度 |
| u_x (空间导数) | 16 | SOH 对每个特征的敏感度 |
| u_t (时间导数) | 1 | SOH 退化速率 |
| **总计** | **35** | |

### 2.3 逻辑：这一步在整体流程中的角色

```
                    autograd 自动微分
                         │
              ┌──────────┼──────────┐
              ▼          ▼          ▼
            u(x,t)    ∂u/∂x     ∂u/∂t
              │          │          │
              │          └────┬─────┘
              │               │
              ▼               ▼
    dynamical_F([x,t, u, u_x, u_t])
              │
              ▼
         F = G(x, u, u_x, u_t)    ← "应有的退化速率"
              │
              ├──── f = u_t - F ──── L_PDE = MSE(f, 0)
              │
              ▼
        如果 f ≈ 0 → 退化过程满足物理方程 du/dt = G(...)
```

**如果没有 L_PDE**：dynamical_F 的输出不参与任何 loss → 梯度不会传给它 → 它永远是初始随机状态 → forward() 中的 F 是无意义的随机数 → f = u_t - random → 整个 PDE 分支就是废的。

### 2.4 物理含义

用电池术语理解 f = u_t - F：

- u_t < 0（SOH 在下降，正常老化），F 也应该是负数（预测到会下降）
  - 如果 F 预测的下降速率 ≈ 实际下降速率 → f ≈ 0 → L_PDE 小 ✓
  - 如果 F 预测上升但实际下降 → f 大 → L_PDE 大 ✗

L_PDE 的物理含义是：**模型内部对退化规律的"自我一致性检查"**。模型不仅要预测 SOH (Solution_u)，还要能用学到的物理规律 (dynamical_F) 解释"为什么退化这么快"。

### 2.5 踩坑记录

| 错误做法 | 现象 | 正确做法 |
|----------|------|----------|
| 只对 f₁ 算 loss | f₂ 不受约束，两个时刻的物理一致性不对称 | f₁ 和 f₂ 各算一份 |
| 用 `f.mean()` 而非 `MSE(f,0)` | 正负 f 会互相抵消！mean([100, -100]) = 0，看起来 loss=0 但其实偏差巨大 | MSE 取平方 → mean([10000, 10000]) = 10000 → 真实反映偏差 |

## 3. 4c: L_mono — 单调性约束 ⭐ <a id="sec3"></a>

> **对应 src/04_loss.py 第 45-47 行**
> **对应原始代码 Model/Model.py 第 254-255 行**

一句话：用 ReLU trick 惩罚"真实 SOH 在下降、但模型预测在上升"的方向错误。**不需要额外的物理方程，只用乘法和 ReLU 就实现了对电池退化不可逆的约束。**

### 3.1 代码块

In [ ]:
def compute_mono_loss(u1, u2, y1, y2, relu):
    """
    L_mono: 单调性约束损失

    公式: L_mono = sum(ReLU((u2 - u1) * (y1 - y2)))

    核心洞察:
        正常退化: y2 < y1 (SOH下降) → y1-y2 > 0
                  预测正确: u2 < u1 → u2-u1 < 0
                  → (u2-u1)*(y1-y2) < 0 → ReLU输出 0 → 无惩罚 ✓

        容量回升: y2 > y1 (SOH反常上升) → y1-y2 < 0
                  预测正确: u2 > u1 → u2-u1 > 0
                  → (u2-u1)*(y1-y2) < 0 → ReLU输出 0 → 无惩罚 ✓

        预测错误: y1 > y2 (SOH下降) 但 u2 > u1 (预测上升)
                  → (u2-u1)*(y1-y2) > 0 → ReLU输出正值 → 惩罚! ✗

    为什么用 sum() 而不是 mean()?
        batch 中每多一个违反单调性的样本，L_mono 就线性增长。
        如果只用 mean()，大量正常样本会"稀释"少量违规的惩罚。
    """
    return relu(torch.mul(u2 - u1, y1 - y2)).sum()

### 3.1.1 逐行详解：ReLU Trick 的数学之美

这是整个 PINN4SOH 中最巧妙的设计。6 行代码，没有引入任何可学习参数，只用乘法和 ReLU 就实现了对复杂物理约束（电池 SOH 不可逆下降）的编码。

---

**第一步：计算预测变化和真实变化**

```python
u2 - u1    # 模型预测的 SOH 变化量
           # < 0 → 模型预测 SOH 下降（正常）
           # > 0 → 模型预测 SOH 上升（可能错误）

y1 - y2    # 真实的 SOH 变化量（注意是 y1-y2，不是 y2-y1！）
           # > 0 → 真实 SOH 下降（正常老化）
           # < 0 → 真实 SOH 上升（容量回升）
```

**第二步：乘积的含义**

| (u₂-u₁) 符号 | (y₁-y₂) 符号 | 乘积符号 | 场景 | 应该? |
|:-----------:|:-----------:|:-------:|------|:----:|
| <0 (预测下降) | >0 (真实下降) | **负** | 正常退化，方向一致 | ✓ 不罚 |
| >0 (预测上升) | <0 (真实上升) | **负** | 容量回升，方向一致 | ✓ 不罚 |
| >0 (预测上升) | >0 (真实下降) | **正** | 方向相反！ | ✗ 惩罚 |
| <0 (预测下降) | <0 (真实上升) | **正** | 方向相反！ | ✗ 惩罚 |

**第三步：ReLU 的选择性惩罚**

```python
ReLU(x) = max(0, x)
```

| 乘积 x | ReLU(x) | 效果 |
|:------:|:-------:|------|
| -0.05 | 0 | 方向正确 → 无惩罚 |
| -0.01 | 0 | 方向正确 → 无惩罚 |
| +0.02 | 0.02 | 方向错误 → 惩罚! |
| +0.10 | 0.10 | 方向错误 → 重罚! |

**为什么 ReLU 而不是直接加惩罚？**

如果用 `|u₂-u₁|` 或 `(u₂-u₁)²` → 会对**所有**变化施加惩罚（包括正常的 SOH 下降），相当于强制模型预测 SOH 不变 = 完全抹杀退化信号。

ReLU 的"单边激活"特性恰好满足需求：只在方向错误时有梯度，方向正确时梯度为 0。

**第四步：sum() vs mean()**

```python
.sum()   # batch 中每个违规样本都和其违规程度成正比地贡献 loss
.mean()  # 大量正常样本会"稀释"少数违规的惩罚
         # 比如 256 个样本中只有 3 个违规，mean 后只剩 3/256 ≈ 1%
```

小批量 + sum = 每个违规样本的惩罚力度不会被稀释。这在 early stopping 场景下尤其重要——如果违规被稀释，early stop 可能永远不会触发（看起来 loss 一直很小）。

### 3.2 语法速查表

| 语法点 | 含义 | 这里为什么用 |
|--------|------|-------------|
| `torch.mul(a, b)` | 逐元素乘法 | 计算 (u₂-u₁)×(y₁-y₂)，每个样本得到一个乘积 |
| `nn.ReLU()` | max(0, x) | 只激活乘积>0的样本（方向错误的），其他输出0 |
| `.sum()` | 所有元素求和 | 每个违规样本都和违规程度成正比贡献 |
| `y1-y2` (不是 y2-y1) | 真实变化的反方向 | 这样乘积的符号才有"方向一致性"的语义 |

**为什么 y1-y2 而不是 y2-y1？**

这是一个精心设计的符号约定：
- 正常退化: y₂<y₁ → y₁-y₂ 是**正数**，代表"退化量"
- 模型预测: u₂<u₁ → u₂-u₁ 是**负数**，代表"SOH 减少了"
- 乘积: 正×负 = **负** → ReLU=0 → 无惩罚

如果把公式写成 `(u₂-u₁)×(y₂-y₁)` → 正常退化时：负×负 = 正 → ReLU>0 → 会错误地惩罚正常退化！

### 3.3 逻辑：L_mono 在训练中的作用

```
每个 batch (256 对):
    ├── 240 对: SOH 正常下降, 模型预测下降 → ReLU=0 → 无贡献
    ├── 10 对: SOH 容量回升, 模型预测上升 → ReLU=0 → 无贡献
    ├── 4 对: SOH 下降, 模型预测上升 → ReLU>0 → 产生惩罚
    └── 2 对: SOH 回升, 模型预测下降 → ReLU>0 → 产生惩罚
                                               ↓
                                    L_mono = sum(6个非零值)
                                               ↓
                                    反向传播: 告诉模型"改掉这 6 个方向错误"
```

**为什么 L_mono 优雅？**
- 不引入任何超参数（不像某些方法需要设置"最大允许上升量"）
- 不修改模型结构（不像某些方法在输出层加 monotonic network）
- 只在方向错误时有梯度，方向正确时完全不影响模型学习
- 自动处理容量回升（真实的物理现象，不应被强制约束）

### 3.4 物理含义

L_mono = 0 并不意味着 SOH 严格单调递减。它只约束"方向"，不约束"幅度"。模型仍可以预测：
- SOH 从 0.90 降到 0.85（正常，方向正确）
- SOH 从 0.85 回升到 0.86（容量回升，如果有真值支持则无惩罚）

只有在**没有真值支持的逆序预测**时才会被惩罚。这比"强制单调递减"的硬约束灵活得多。

### 3.5 踩坑记录

| 错误做法 | 现象 | 正确做法 |
|----------|------|----------|
| 用 `mean()` 而非 `sum()` | 违规被正常样本稀释，模型慢慢学会"偶尔违规也没事" | 用 `sum()` 保持惩罚力度 |
| 公式写成 `(u2-u1)*(y2-y1)` | 正常退化被错误惩罚，模型不敢预测下降 → SOH 全平 | 用 `(u2-u1)*(y1-y2)`，注意是 y1-y2 |
| 用 `abs()` 而非 `ReLU()` | 所有变化都被惩罚 → 模型学会预测 SOH 完全不变 | ReLU 只激活方向错误 |

## 4. 4d: α, β 参数分析 <a id="sec4"></a>

> **对应 src/04_loss.py 第 50-57 行**

一句话：α 控制"物理约束有多强"，β 控制"单调性约束有多严"。两者的平衡决定了模型是偏物理还是偏数据。

### 4.1 代码块

In [ ]:
def compute_total_loss(u1, u2, f1, f2, y1, y2,
                       loss_func, relu, alpha=0.7, beta=0.2):
    """
    三项 Loss 汇总

    公式: L_total = L_data + alpha * L_PDE + beta * L_mono

    alpha 的含义:
        alpha=0   → 纯数据驱动 MLP（无视物理）
        alpha=0.7 → 论文推荐：物理和数据平衡
        alpha=2.0 → 强物理约束，可能欠拟合数据

    beta 的含义:
        beta=0   → 不约束单调性
        beta=0.2 → 论文推荐：轻微约束
        beta=2.0 → 严苛约束，可能压制真实的容量回升
    """
    loss1 = compute_data_loss(u1, u2, y1, y2, loss_func)
    loss2 = compute_pde_loss(f1, f2, loss_func)
    loss3 = compute_mono_loss(u1, u2, y1, y2, relu)
    total = loss1 + alpha * loss2 + beta * loss3
    return total, loss1, loss2, loss3

### 4.1.1 参数影响实验

以下是用未训练模型 + 随机数据跑的不同 α, β 组合下三个 loss 的数值和 total:

In [ ]:
import sys, os
sys.path.insert(0, r'../src')
import importlib.util
def _load(fname, mname):
    spec = importlib.util.spec_from_file_location(mname, os.path.join(r'../src', fname))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

_model = _load('03_model.py', 'model_03')
PINN = _model.PINN
device = _model.device

pinn = PINN(F_layers_num=3, F_hidden_dim=60)
torch.manual_seed(42)

# 模拟数据: 8 个样本，正常退化 + 少量容量回升
x1 = torch.randn(8, 17).to(device)
x2 = torch.randn(8, 17).to(device)
y1 = torch.rand(8, 1).to(device) * 0.2 + 0.8
y2 = y1 - torch.rand(8, 1).to(device) * 0.02
y2 = torch.clamp(y2, min=0.6)
y2[0] = y1[0] + 0.01  # 容量回升

u1, f1 = pinn.forward(x1)
u2, f2 = pinn.forward(x2)

alphas = [0.0, 0.1, 0.5, 0.7, 1.0, 2.0]
betas  = [0.0, 0.1, 0.2, 0.5, 1.0, 2.0]

print(f"{'alpha':<8} {'beta':<8} {'L_data':<12} {'L_PDE':<12} {'L_mono':<12} {'L_total':<12}")
print("-" * 64)
for alpha in [0.0, 0.7, 2.0]:
    for beta in [0.0, 0.2, 2.0]:
        L1 = 0.5 * pinn.loss_func(u1, y1) + 0.5 * pinn.loss_func(u2, y2)
        f_target = torch.zeros_like(f1)
        L2 = 0.5 * pinn.loss_func(f1, f_target) + 0.5 * pinn.loss_func(f2, f_target)
        L3 = pinn.relu(torch.mul(u2-u1, y1-y2)).sum()
        total = L1 + alpha * L2 + beta * L3
        print(f"{alpha:<8.1f} {beta:<8.1f} {L1.item():<12.6f} {L2.item():<12.6f} {L3.item():<12.6f} {total.item():<12.6f}")

print()
print("观察:")
print("  1. L_PDE 通常比 L_data 大很多 → alpha<1.0 防止 PDE 主导")
print("  2. L_mono 相对很小 → beta 不需要太大就能起作用")
print("  3. alpha=0 → 纯数据驱动; beta=0 → 无单调性约束")

### 4.2 设计决策

| 参数 | 论文值 | 物理含义 | 调大后果 | 调小后果 |
|------|:-----:|----------|---------|---------|
| α | 0.7 | PDE 约束权重 | 模型死守物理方程，不拟合数据 → 欠拟合 | 退化为普通 MLP → 在少样本上过拟合 |
| β | 0.2 | 单调性约束权重 | 过度压制容量回升预测 → MAE 在回升区更大 | 可能出现 SOH 上升预测 → 物理不可信 |

**论文为什么选 α=0.7, β=0.2？**

```
L_PDE 的量级 (0.01~1.0) 通常远大于 L_mono (0.0001~0.01)
→ alpha < 1.0 防止 PDE loss 主导总 loss
→ beta < 1.0 防止 monotonicity 在早期压制模型探索

经验: L_data ≈ 0.01, L_PDE ≈ 0.1, L_mono ≈ 0.001 时(良好训练后)
      alpha=0.7 → PDE contribution = 0.07 ≈ 7×L_data
      beta=0.2  → mono contribution = 0.0002 ≈ 0.02×L_data
      → PDE 约束主导但不压倒数据拟合
      → 单调性约束轻量但持续存在
```

## 5. 完整训练一步: train_one_epoch <a id="sec5"></a>

> **对应 src/04_loss.py 第 60-81 行**
> **对应原始代码 Model/Model.py 第 237-275 行**

一句话：把三个 Loss + 双 optimizer 的反向传播串联成一次完整的训练迭代。

### 5.1 代码块

In [ ]:
class AverageMeter:
    """追踪训练过程中的平均值"""
    def __init__(self):
        self.reset()
    def reset(self):
        self.val = 0.0; self.avg = 0.0; self.sum = 0.0; self.count = 0
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


def train_one_epoch(pinn, dataloader, optimizer1, optimizer2,
                    alpha=0.7, beta=0.2, epoch=1, verbose=True):
    """
    执行一个 epoch 的训练

    双 Optimizer 设计:
        optimizer1 → solution_u (SOH预测网络)
        optimizer2 → dynamical_F (退化动力学网络)
        独立优化允许两个子网络用不同的学习率
    """
    pinn.train()
    loss1_meter = AverageMeter()
    loss2_meter = AverageMeter()
    loss3_meter = AverageMeter()

    for iter_idx, (x1, x2, y1, y2) in enumerate(dataloader):
        x1, x2 = x1.to(device), x2.to(device)
        y1, y2 = y1.to(device), y2.to(device)

        # 1. 前向传播
        u1, f1 = pinn.forward(x1)
        u2, f2 = pinn.forward(x2)

        # 2. 计算三个 Loss
        L1 = 0.5 * pinn.loss_func(u1, y1) + 0.5 * pinn.loss_func(u2, y2)
        f_target = torch.zeros_like(f1)
        L2 = 0.5 * pinn.loss_func(f1, f_target) + 0.5 * pinn.loss_func(f2, f_target)
        L3 = pinn.relu(torch.mul(u2 - u1, y1 - y2)).sum()
        total = L1 + alpha * L2 + beta * L3

        # 3. 反向传播 (双 optimizer)
        optimizer1.zero_grad()
        optimizer2.zero_grad()
        total.backward()
        optimizer1.step()
        optimizer2.step()

        # 4. 记录
        loss1_meter.update(L1.item())
        loss2_meter.update(L2.item())
        loss3_meter.update(L3.item())

        if verbose and (iter_idx + 1) % 10 == 0:
            print(f"[epoch:{epoch} iter:{iter_idx+1}] "
                  f"data:{L1.item():.6f} PDE:{L2.item():.6f} mono:{L3.item():.6f}")

    return loss1_meter.avg, loss2_meter.avg, loss3_meter.avg

### 5.1.1 逐行详解：反向传播流

**最关键的 4 行**：

```python
optimizer1.zero_grad()    # 清零 solution_u 的梯度
optimizer2.zero_grad()    # 清零 dynamical_F 的梯度
total.backward()          # 反向传播: total → 两个子网络都获得梯度
optimizer1.step()         # 更新 solution_u 参数
optimizer2.step()         # 更新 dynamical_F 参数
```

**反向传播的梯度流向**：

```
                    total = L1 + α·L2 + β·L3
                      │
            ┌─────────┼─────────┐
            ▼         ▼         ▼
          L1        α·L2      β·L3
            │         │         │
            ▼         ▼         ▼
        solution_u  BOTH!    solution_u
        (via u₁,u₂) (via f)  (via u₁,u₂)
            │         │         │
            │         ▼         │
            │    dynamical_F    │
            │    (via F)        │
            ▼         ▼         ▼
        optimizer1  optimizer2  optimizer1
```

**关键观察**：L_PDE 的梯度同时流向 solution_u（通过 u_t, u_x）和 dynamical_F（通过 F）。这意味着两个子网络通过 PDE loss 互相约束：
- solution_u 被约束学到"容易被 F 解释"的 SOH 轨迹
- dynamical_F 被约束学到"能解释 u 的变化"的退化规律

这种**双向约束**是 PINN 比"先训练 encoder 再固定 encoder 训 F"更好的原因。

### 5.2 完整数据流

```
DataLoader (x1, x2, y1, y2)  ─── 每个 batch 256 对
    │
    ├─ x1 → PINN.forward(x1) → u1, f1
    ├─ x2 → PINN.forward(x2) → u2, f2
    │
    ├─ L_data  = ½MSE(u1,y1) + ½MSE(u2,y2)
    ├─ L_PDE   = ½MSE(f1,0)  + ½MSE(f2,0)
    ├─ L_mono  = ΣReLU((u2-u1)*(y1-y2))
    └─ L_total = L_data + 0.7·L_PDE + 0.2·L_mono
         │
         ▼
    total.backward() ─── 梯度流向 solution_u + dynamical_F
         │
         ▼
    optimizer1.step() ──── optimizer2.step()
         │
         ▼
    下一个 batch...
```

---

### 与原始代码的对应关系

| 原始代码 | clean_code | 变化说明 |
|----------|-----------|----------|
| `self.loss_func` (nn.MSELoss) | `loss_func` 参数 | 解耦，允许传入其他 loss 函数 |
| `self.relu` (nn.ReLU) | `relu` 参数 | 同上 |
| `self.alpha` / `self.beta` | `alpha` / `beta` 参数 | 从 args 中独立出来 |
| 内联在 `train_one_epoch` | 三个独立函数 + train_one_epoch | 每个 Loss 可单独测试和理解 |
| `AverageMeter` 在 util.py | 内联在 04_loss.py | 减少跨文件依赖 |

---

> **模块 4 总结**：三项 Loss 是 PINN 区别于普通 MLP 的核心。L_data 保证预测准确，L_PDE 注入物理规律，L_mono 约束退化方向。α 和 β 是平衡这三个目标的"旋钮"——调得好，模型在小样本电池数据上才能学到可靠的退化规律。